# Defensive ETF Rotation - Research Notebook

**Source**: QC Strategy Library - Conditional Sector Rotation

**Concept**: RSI-based conditional rotation among 9 ETFs (equity, leveraged, inverse, volatility, bonds).
Uses SPY 200-SMA as regime filter, then nested RSI thresholds to select the best target.
100% concentrated single-position allocation with daily signal evaluation.

In [ ]:
from AlgorithmImports import *
qb = QuantBook()

# Define universe
tickers = ["SPY", "QQQ", "TQQQ", "UVXY", "TECL", "SPXL", "SQQQ", "TECS", "BSV"]
for t in tickers:
    qb.AddEquity(t, Resolution.Daily)

print(f"Universe: {tickers}")
print(f"Strategy: Regime-filtered RSI rotation")

## Strategy Logic

1. **Regime filter**: SPY price vs 200-day SMA (bull vs bear)
2. **Bull regime**: RSI overbought on QQQ/SPY -> UVXY hedge, else TQQQ
3. **Bear regime**: Multi-level RSI conditions selecting from TECL, SPXL, TECS, UVXY, BSV
4. **Concentrated position**: 100% in single ETF at all times
5. **Daily signal**: Re-evaluated every trading day

In [ ]:
# Fetch historical data for analysis
spy_hist = qb.History("SPY", timedelta(252 * 14), Resolution.Daily)
qqq_hist = qb.History("QQQ", timedelta(252 * 14), Resolution.Daily)
print(f"SPY data shape: {spy_hist.shape}")
print(f"QQQ data shape: {qqq_hist.shape}")

# Calculate regime indicator
close_spy = spy_hist["close"]
sma_200 = close_spy.rolling(200).mean()
regime = (close_spy > sma_200).astype(int)

print(f"\nBull regime (SPY > 200-SMA): {regime.mean():.1%} of the time")
print(f"Bear regime (SPY < 200-SMA): {1-regime.mean():.1%} of the time")

In [ ]:
# Analyze RSI distribution for signal thresholds
close_qqq = qqq_hist["close"]
delta = close_qqq.diff()
gain = delta.where(delta > 0, 0).rolling(10).mean()
loss = (-delta.where(delta < 0, 0)).rolling(10).mean()
rs = gain / loss
rsi_qqq = 100 - (100 / (1 + rs))

print("QQQ 10-day RSI statistics:")
print(f"  Mean: {rsi_qqq.mean():.1f}")
print(f"  Median: {rsi_qqq.median():.1f}")
print(f"  > 81 threshold: {(rsi_qqq > 81).mean():.1%} of days")
print(f"  < 30 threshold: {(rsi_qqq < 30).mean():.1%} of days")
print(f"  Current: {rsi_qqq.iloc[-1]:.1f}")

## Key Observations

- **Leveraged ETFs** (TQQQ, TECL, SPXL, TECS, SQQQ) amplify both returns and losses
- **Volatility product** (UVXY) tends to decay over time but spikes during stress
- **RSI thresholds** are asymmetric: 80+ for overbought vs 30- for oversold
- **Regime filter** prevents holding leveraged longs during bear markets

### Strengths
- Clear bull/bear regime separation via 200-SMA
- Uses multiple RSI signals across correlated assets for robustness
- UVXY allocation provides tail-risk hedging during overbought conditions
- Fully invested at all times (no cash drag)

### Risks
- 100% concentrated position has high volatility
- Leveraged ETFs suffer from volatility decay in choppy markets
- UVXY contango drag erodes returns during low-vol regimes
- Frequent daily rotation generates high turnover and slippage
- RSI thresholds are fixed and not adaptive to market structure changes

In [ ]:
# Signal distribution analysis
print("=" * 60)
print("SIGNAL TRIGGER ANALYSIS")
print("=" * 60)
print()
print("Bull regime signals (SPY > 200-SMA):")
print(f"  TQQQ (default long):     ~{(regime[-252*5:].sum()/len(regime[-252*5:])):.0%} of bull days")
print(f"  UVXY (QQQ RSI > 81):     Rare event")
print(f"  UVXY (SPY RSI > 80):     Rare event")
print()
print("Bear regime signals (SPY < 200-SMA):")
print(f"  TECL (TQQQ RSI < 30):    Oversold bounce")
print(f"  SPXL (SPY RSI < 30):     Deep oversold")
print(f"  UVXY (74 < RSI < 84):    Vol trending up")
print(f"  TECS/BSV (max RSI):      Defensive fallback")

In [ ]:
# Backtest results placeholder
print("=" * 50)
print("BACKTEST RESULTS (14-year: 2011-2025)")
print("=" * 50)
print("Run backtest via QC MCP to populate metrics:")
print("  create_compile -> create_backtest -> read_backtest")
print()
print("Expected characteristics:")
print("  - CAGR: ~15-25% (leveraged positions)")
print("  - Max Drawdown: ~30-50% (leveraged drawdowns)")
print("  - Sharpe: ~0.6-1.0")
print("  - Rebalance frequency: Daily")
print("  - Avg positions: 1 (concentrated)")
print("  - Universe: 9 ETFs (equity, leveraged, inverse, vol, bonds)")